# 🧠 Vectores de Refusal en LLMs: Una Guía Visual Completa

Este notebook explica desde cero cómo funcionan los **vectores de rechazo (refusal)** en modelos de lenguaje grandes (LLMs), incluyendo:

1. **Fundamentos**: Cómo los tokens se convierten en vectores
2. **Arquitectura**: Flujo de información en un Transformer
3. **Emergencia del Refusal**: Por qué y cómo los modelos "aprenden" a rechazar
4. **Extracción**: Cómo encontrar la dirección de refusal
5. **Manipulación**: Técnicas para modificar el comportamiento

---

In [ ]:
# Instalación de dependencias
# !pip install numpy matplotlib ipywidgets plotly

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

np.random.seed(42)
print("✅ Librerías cargadas")

---
# 📖 Parte 1: De Texto a Vectores

## 1.1 ¿Qué es un Token?

Los LLMs no ven texto como humanos. Ven **tokens**: fragmentos de texto convertidos a números.

```
"Hello world" → [15496, 995]  (dos tokens)
"Hola mundo"  → [39, 5765, 29958]  (puede ser diferente según el tokenizer)
```

Cada token es un **índice** en un vocabulario de ~50,000-100,000 palabras/subpalabras.

In [ ]:
def visualizar_tokenizacion():
    """Visualiza el proceso de tokenización"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Panel 1: Texto original
    ax1 = axes[0]
    ax1.text(0.5, 0.5, '"How to make a bomb?"', 
             fontsize=16, ha='center', va='center',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    ax1.axis('off')
    ax1.set_title('1. Texto Original', fontsize=14, fontweight='bold')
    
    # Panel 2: Tokens
    ax2 = axes[1]
    tokens = ['How', ' to', ' make', ' a', ' bomb', '?']
    colors = plt.cm.Set3(np.linspace(0, 1, len(tokens)))
    
    for i, (tok, col) in enumerate(zip(tokens, colors)):
        ax2.add_patch(plt.Rectangle((i*0.15 + 0.05, 0.4), 0.13, 0.2, 
                                     facecolor=col, edgecolor='black'))
        ax2.text(i*0.15 + 0.115, 0.5, tok, ha='center', va='center', fontsize=10)
        ax2.text(i*0.15 + 0.115, 0.3, f'id:{1000+i*123}', ha='center', va='center', fontsize=8)
    
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.axis('off')
    ax2.set_title('2. Tokenización', fontsize=14, fontweight='bold')
    
    # Panel 3: IDs numéricos
    ax3 = axes[2]
    ids = [1000, 1123, 1246, 1369, 1492, 1615]
    ax3.text(0.5, 0.5, f'[{ids[0]}, {ids[1]}, {ids[2]},\n {ids[3]}, {ids[4]}, {ids[5]}]', 
             fontsize=14, ha='center', va='center', family='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax3.set_xlim(0, 1)
    ax3.set_ylim(0, 1)
    ax3.axis('off')
    ax3.set_title('3. IDs Numéricos', fontsize=14, fontweight='bold')
    
    plt.suptitle('Proceso de Tokenización', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

visualizar_tokenizacion()

## 1.2 De Token ID a Embedding Vector

Cada token ID se convierte en un **vector de alta dimensión** mediante una **tabla de embeddings**.

### La Matemática:

$$\text{embedding}(\text{token}_i) = E[\text{token}_i] \in \mathbb{R}^{d}$$

Donde:
- $E$ es la matriz de embeddings de tamaño $(V \times d)$
- $V$ = tamaño del vocabulario (~50,000)
- $d$ = dimensión del embedding (768 en GPT-2, 4096 en LLaMA-7B, etc.)

In [ ]:
def visualizar_embedding_lookup():
    """Visualiza cómo un token ID se convierte en un vector embedding"""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Simular matriz de embeddings pequeña para visualización
    vocab_size = 8
    embed_dim = 6
    E = np.random.randn(vocab_size, embed_dim) * 0.5
    
    # Panel 1: Token ID
    ax1 = axes[0]
    ax1.text(0.5, 0.7, 'Token: "bomb"', fontsize=14, ha='center', va='center')
    ax1.text(0.5, 0.4, 'ID: 5', fontsize=20, ha='center', va='center',
             bbox=dict(boxstyle='round', facecolor='coral', alpha=0.8))
    ax1.arrow(0.5, 0.25, 0, -0.15, head_width=0.05, head_length=0.03, fc='black', ec='black')
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    ax1.axis('off')
    ax1.set_title('1. Token ID', fontsize=12, fontweight='bold')
    
    # Panel 2: Matriz de embeddings
    ax2 = axes[1]
    im = ax2.imshow(E, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
    ax2.axhline(y=4.5, color='yellow', linewidth=3, linestyle='--')
    ax2.axhline(y=5.5, color='yellow', linewidth=3, linestyle='--')
    
    # Resaltar fila 5
    for j in range(embed_dim):
        ax2.add_patch(plt.Rectangle((j-0.5, 4.5), 1, 1, fill=False, 
                                     edgecolor='yellow', linewidth=2))
    
    ax2.set_xlabel('Dimensión del Embedding', fontsize=11)
    ax2.set_ylabel('Token ID', fontsize=11)
    ax2.set_yticks(range(vocab_size))
    ax2.set_xticks(range(embed_dim))
    ax2.set_title(f'2. Matriz E ({vocab_size}×{embed_dim})\n(fila 5 resaltada)', 
                  fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax2, shrink=0.8)
    
    # Panel 3: Vector resultado
    ax3 = axes[2]
    embedding_vector = E[5]
    bars = ax3.bar(range(embed_dim), embedding_vector, color='coral', edgecolor='black')
    ax3.axhline(y=0, color='black', linewidth=0.5)
    ax3.set_xlabel('Dimensión', fontsize=11)
    ax3.set_ylabel('Valor', fontsize=11)
    ax3.set_title(f'3. Vector Embedding\ne("bomb") ∈ ℝ^{embed_dim}', 
                  fontsize=12, fontweight='bold')
    ax3.set_xticks(range(embed_dim))
    
    # Añadir valores encima de las barras
    for i, v in enumerate(embedding_vector):
        ax3.text(i, v + 0.05 if v >= 0 else v - 0.1, f'{v:.2f}', 
                ha='center', va='bottom' if v >= 0 else 'top', fontsize=8)
    
    plt.suptitle('Embedding Lookup: E[token_id] → vector', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    return E[5]

embedding_ejemplo = visualizar_embedding_lookup()

## 1.3 El Espacio de Embeddings

Los embeddings **no son aleatorios**. Después del entrenamiento, capturan **relaciones semánticas**:

- Palabras similares → vectores cercanos
- Analogías: `king - man + woman ≈ queen`

### Conceptos clave:

| Concepto | Significado |
|----------|-------------|
| **Distancia** | Similitud semántica |
| **Dirección** | Concepto o atributo |
| **Región** | Cluster de conceptos relacionados |

In [ ]:
def visualizar_espacio_embeddings():
    """Visualiza el espacio de embeddings con clusters semánticos"""
    fig = plt.figure(figsize=(14, 6))
    
    # Panel 1: Vista 2D del espacio
    ax1 = fig.add_subplot(121)
    
    # Crear clusters simulados
    np.random.seed(42)
    
    # Cluster: palabras peligrosas
    dangerous = np.random.randn(15, 2) * 0.3 + np.array([2, 2])
    dangerous_words = ['bomb', 'weapon', 'explosive', 'attack', 'violence',
                       'kill', 'harm', 'destroy', 'danger', 'threat',
                       'hack', 'steal', 'illegal', 'drug', 'poison']
    
    # Cluster: palabras neutrales
    neutral = np.random.randn(15, 2) * 0.4 + np.array([-1, -1])
    neutral_words = ['hello', 'help', 'code', 'write', 'explain',
                     'learn', 'teach', 'create', 'design', 'build',
                     'think', 'solve', 'analyze', 'improve', 'develop']
    
    # Cluster: palabras de rechazo
    refusal = np.random.randn(8, 2) * 0.25 + np.array([0.5, 3])
    refusal_words = ['cannot', 'unable', 'sorry', 'refuse',
                     'inappropriate', 'harmful', 'dangerous', 'unethical']
    
    # Plotear
    ax1.scatter(dangerous[:, 0], dangerous[:, 1], c='red', s=100, alpha=0.7, label='Peligrosas')
    ax1.scatter(neutral[:, 0], neutral[:, 1], c='green', s=100, alpha=0.7, label='Neutrales')
    ax1.scatter(refusal[:, 0], refusal[:, 1], c='purple', s=100, alpha=0.7, label='Rechazo')
    
    # Etiquetas
    for i, word in enumerate(dangerous_words[:5]):
        ax1.annotate(word, dangerous[i], fontsize=8, alpha=0.8)
    for i, word in enumerate(neutral_words[:5]):
        ax1.annotate(word, neutral[i], fontsize=8, alpha=0.8)
    for i, word in enumerate(refusal_words[:4]):
        ax1.annotate(word, refusal[i], fontsize=8, alpha=0.8)
    
    # Dirección de "peligro" a "rechazo"
    center_danger = np.mean(dangerous, axis=0)
    center_refusal = np.mean(refusal, axis=0)
    ax1.annotate('', xy=center_refusal, xytext=center_danger,
                arrowprops=dict(arrowstyle='->', color='orange', lw=3))
    ax1.text((center_danger[0]+center_refusal[0])/2 + 0.3, 
             (center_danger[1]+center_refusal[1])/2,
             'Dirección\nde refusal?', fontsize=10, color='orange')
    
    ax1.set_xlabel('Dimensión 1')
    ax1.set_ylabel('Dimensión 2')
    ax1.set_title('Espacio de Embeddings (simplificado a 2D)', fontweight='bold')
    ax1.legend()
    ax1.set_aspect('equal')
    
    # Panel 2: Analogía vectorial
    ax2 = fig.add_subplot(122)
    
    # Vectores para analogía
    king = np.array([1, 2])
    man = np.array([0.5, 0.5])
    woman = np.array([-0.5, 0.5])
    queen = king - man + woman  # La analogía
    
    # Plotear puntos
    points = {'king': king, 'man': man, 'woman': woman, 'queen': queen}
    colors = {'king': 'blue', 'man': 'green', 'woman': 'pink', 'queen': 'purple'}
    
    for name, point in points.items():
        ax2.scatter(*point, c=colors[name], s=200, zorder=5)
        ax2.annotate(name, point + 0.1, fontsize=12, fontweight='bold')
    
    # Flechas de la analogía
    ax2.annotate('', xy=woman, xytext=man, 
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))
    ax2.annotate('', xy=queen, xytext=king,
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))
    ax2.annotate('', xy=king, xytext=man,
                arrowprops=dict(arrowstyle='->', color='blue', lw=2, ls='--'))
    ax2.annotate('', xy=queen, xytext=woman,
                arrowprops=dict(arrowstyle='->', color='purple', lw=2, ls='--'))
    
    ax2.text(0, -0.5, 'king - man + woman ≈ queen', fontsize=12, 
             ha='center', style='italic',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax2.set_xlabel('Dimensión 1')
    ax2.set_ylabel('Dimensión 2')
    ax2.set_title('Analogías = Aritmética Vectorial', fontweight='bold')
    ax2.set_xlim(-1.5, 2)
    ax2.set_ylim(-1, 3)
    ax2.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()

visualizar_espacio_embeddings()

### 💡 Insight Clave

> **Si conceptos = direcciones en el espacio de embeddings, entonces "refusal" también es una dirección.**
>
> Encontrar esa dirección y removerla = eliminar el comportamiento de rechazo.

---

# 🏗️ Parte 2: Arquitectura Transformer

## 2.1 Vista General: El Flujo de Información

Un Transformer procesa tokens en **capas sucesivas**. Cada capa transforma las representaciones:

```
Input Tokens → Embedding → [Capa 1] → [Capa 2] → ... → [Capa N] → Output Logits
```

### Componentes de cada capa:

1. **Attention**: "¿A qué otros tokens debo prestar atención?"
2. **MLP/FFN**: "¿Cómo proceso esta información?"
3. **Residual Connections**: Conexiones que suman el input al output

### Dimensiones típicas:

| Modelo | Capas (N) | Dimensión (d) | Atención (heads) |
|--------|-----------|---------------|------------------|
| GPT-2 Small | 12 | 768 | 12 |
| GPT-2 XL | 48 | 1600 | 25 |
| LLaMA-7B | 32 | 4096 | 32 |
| LLaMA-70B | 80 | 8192 | 64 |

In [ ]:
def visualizar_transformer_arquitectura():
    """Visualiza la arquitectura de un Transformer y el flujo de activaciones"""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Panel 1: Arquitectura de capas
    ax1 = axes[0]
    ax1.set_xlim(0, 10)
    ax1.set_ylim(0, 12)
    ax1.axis('off')
    
    # Dibujar capas
    n_layers = 4
    layer_colors = plt.cm.Blues(np.linspace(0.3, 0.8, n_layers))
    
    # Input
    ax1.add_patch(plt.Rectangle((3, 0.5), 4, 0.8, facecolor='lightgreen', edgecolor='black'))
    ax1.text(5, 0.9, 'Input Embeddings\n[seq_len × d]', ha='center', va='center', fontsize=9)
    
    y_pos = 1.8
    for i in range(n_layers):
        # Capa completa
        ax1.add_patch(plt.Rectangle((1, y_pos), 8, 2), 
                      facecolor=layer_colors[i], edgecolor='black', alpha=0.3)
        ax1.text(0.5, y_pos + 1, f'Capa {i+1}', ha='center', va='center', fontsize=10, fontweight='bold', rotation=90)
        
        # Attention
        ax1.add_patch(plt.Rectangle((1.5, y_pos + 0.2), 3, 0.7, facecolor='coral', edgecolor='black'))
        ax1.text(3, y_pos + 0.55, 'Attention', ha='center', va='center', fontsize=8)
        
        # Add & Norm
        ax1.add_patch(plt.Rectangle((1.5, y_pos + 1.1), 3, 0.4, facecolor='lightyellow', edgecolor='black'))
        ax1.text(3, y_pos + 1.3, '+ LayerNorm', ha='center', va='center', fontsize=7)
        
        # MLP
        ax1.add_patch(plt.Rectangle((5.5, y_pos + 0.2), 3, 0.7, facecolor='lightblue', edgecolor='black'))
        ax1.text(7, y_pos + 0.55, 'MLP (FFN)', ha='center', va='center', fontsize=8)
        
        # Add & Norm
        ax1.add_patch(plt.Rectangle((5.5, y_pos + 1.1), 3, 0.4, facecolor='lightyellow', edgecolor='black'))
        ax1.text(7, y_pos + 1.3, '+ LayerNorm', ha='center', va='center', fontsize=7)
        
        # Flecha residual
        ax1.annotate('', xy=(3, y_pos + 1.6), xytext=(3, y_pos + 0.1),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
        ax1.annotate('', xy=(7, y_pos + 1.6), xytext=(7, y_pos + 0.1),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
        
        y_pos += 2.3
    
    # Output
    ax1.add_patch(plt.Rectangle((3, y_pos), 4, 0.8, facecolor='lightyellow', edgecolor='black'))
    ax1.text(5, y_pos + 0.4, 'Output Logits\n[seq_len × vocab]', ha='center', va='center', fontsize=9)
    
    ax1.set_title('Arquitectura Transformer (simplificada)', fontsize=14, fontweight='bold')
    
    # Panel 2: Flujo de un vector a través de las capas
    ax2 = axes[1]
    
    # Simular evolución de un vector
    n_layers_sim = 6
    d_sim = 50  # Dimensiones simuladas
    
    # Vector inicial (embedding de "bomb")
    np.random.seed(42)
    h_0 = np.random.randn(d_sim) * 0.5
    
    # Simular transformaciones por capa
    activations = [h_0]
    for layer in range(n_layers_sim):
        # Simulación simplificada: cada capa modifica el vector
        W = np.random.randn(d_sim, d_sim) * 0.1
        h_new = activations[-1] + np.tanh(W @ activations[-1]) * 0.3
        activations.append(h_new)
    
    activations = np.array(activations)
    
    # Visualizar como heatmap
    im = ax2.imshow(activations.T, aspect='auto', cmap='RdBu', vmin=-2, vmax=2)
    ax2.set_xlabel('Capa', fontsize=12)
    ax2.set_ylabel('Dimensión del vector', fontsize=12)
    ax2.set_xticks(range(n_layers_sim + 1))
    ax2.set_xticklabels(['Emb'] + [f'L{i+1}' for i in range(n_layers_sim)])
    ax2.set_title('Evolución de h (hidden state) a través de capas\n(cada columna = vector en esa capa)', 
                  fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax2, label='Valor de activación')
    
    # Resaltar algunas dimensiones que "se activan"
    ax2.axhline(y=10, color='yellow', linewidth=2, linestyle='--', alpha=0.7)
    ax2.axhline(y=25, color='yellow', linewidth=2, linestyle='--', alpha=0.7)
    ax2.text(n_layers_sim + 0.5, 10, '← dim. que se\n   "enciende"', fontsize=8, va='center')
    
    plt.tight_layout()
    plt.show()
    
    return activations

activations = visualizar_transformer_arquitectura()

## 2.2 El "Residual Stream": Concepto Clave

### ¿Por qué importan las conexiones residuales?

En un Transformer, la información fluye a través del **residual stream**:

$$h^{(l+1)} = h^{(l)} + \text{Attention}(h^{(l)}) + \text{MLP}(h^{(l)})$$

Esto significa que:
1. **Cada capa SUMA** su contribución al vector
2. El vector original **nunca se pierde**
3. Podemos interpretar cada capa como "añadiendo información"

### Implicación para Refusal:

> **El "refusal" puede verse como una dirección que se SUMA al residual stream en ciertas capas.**
>
> Si identificamos esa dirección, podemos RESTARLA.

In [ ]:
def visualizar_residual_stream():
    """Visualiza el concepto de residual stream y cómo se acumula información"""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Panel 1: Residual stream como suma
    ax1 = axes[0]
    ax1.set_xlim(0, 10)
    ax1.set_ylim(0, 8)
    ax1.axis('off')
    
    # Dibujar el flujo
    y_positions = [1, 2.5, 4, 5.5, 7]
    labels = ['h₀ (embed)', 'h₁ = h₀ + Δ₁', 'h₂ = h₁ + Δ₂', 'h₃ = h₂ + Δ₃', 'h_final']
    colors = ['lightgreen', 'lightblue', 'lightblue', 'lightblue', 'lightyellow']
    
    for i, (y, label, color) in enumerate(zip(y_positions, labels, colors)):
        ax1.add_patch(plt.Rectangle((2, y-0.3), 6, 0.6, facecolor=color, edgecolor='black'))
        ax1.text(5, y, label, ha='center', va='center', fontsize=10, family='monospace')
        
        if i < len(y_positions) - 1:
            # Flecha principal (residual)
            ax1.annotate('', xy=(5, y_positions[i+1]-0.3), xytext=(5, y+0.3),
                        arrowprops=dict(arrowstyle='->', color='blue', lw=2))
            # Delta (contribución de la capa)
            if i > 0:
                ax1.annotate('', xy=(7.5, y_positions[i+1]-0.3), xytext=(8.5, y+0.5),
                            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
                ax1.text(9, y+0.3, f'Δ{i+1}', fontsize=9, color='red')
    
    ax1.text(5, 0.3, 'Residual Stream: h se va acumulando', ha='center', fontsize=11, fontweight='bold')
    ax1.set_title('Concepto de Residual Stream', fontsize=12, fontweight='bold')
    
    # Panel 2: Descomposición vectorial
    ax2 = axes[1]
    
    np.random.seed(123)
    # Simular vectores 2D para visualización
    h0 = np.array([0.5, 0.3])
    delta1 = np.array([0.8, 0.2])
    delta2 = np.array([0.2, 0.9])
    delta3 = np.array([-0.3, 0.4])  # Esta podría ser la "dirección de refusal"
    
    h1 = h0 + delta1
    h2 = h1 + delta2
    h3 = h2 + delta3
    
    # Plotear acumulación
    ax2.arrow(0, 0, h0[0], h0[1], head_width=0.08, head_length=0.05, fc='green', ec='green', lw=2)
    ax2.arrow(h0[0], h0[1], delta1[0], delta1[1], head_width=0.08, head_length=0.05, fc='blue', ec='blue', lw=2)
    ax2.arrow(h1[0], h1[1], delta2[0], delta2[1], head_width=0.08, head_length=0.05, fc='blue', ec='blue', lw=2)
    ax2.arrow(h2[0], h2[1], delta3[0], delta3[1], head_width=0.08, head_length=0.05, fc='red', ec='red', lw=2)
    
    # Vector final
    ax2.arrow(0, 0, h3[0], h3[1], head_width=0.1, head_length=0.06, fc='purple', ec='purple', lw=3, alpha=0.5)
    
    # Etiquetas
    ax2.text(h0[0]/2-0.1, h0[1]/2+0.1, 'h₀', fontsize=10, color='green')
    ax2.text(h0[0]+delta1[0]/2+0.1, h0[1]+delta1[1]/2, 'Δ₁', fontsize=10, color='blue')
    ax2.text(h1[0]+delta2[0]/2+0.1, h1[1]+delta2[1]/2, 'Δ₂', fontsize=10, color='blue')
    ax2.text(h2[0]+delta3[0]/2-0.2, h2[1]+delta3[1]/2+0.1, 'Δ₃\n(refusal?)', fontsize=9, color='red')
    ax2.text(h3[0]+0.1, h3[1]+0.1, 'h_final', fontsize=10, color='purple')
    
    ax2.set_xlim(-0.5, 2.5)
    ax2.set_ylim(-0.3, 2.5)
    ax2.set_aspect('equal')
    ax2.set_xlabel('Dimensión 1')
    ax2.set_ylabel('Dimensión 2')
    ax2.set_title('h_final = h₀ + Δ₁ + Δ₂ + Δ₃', fontsize=12, fontweight='bold')
    ax2.axhline(y=0, color='gray', linewidth=0.5)
    ax2.axvline(x=0, color='gray', linewidth=0.5)
    
    # Panel 3: ¿Qué pasa si removemos Δ₃?
    ax3 = axes[2]
    
    h3_sin_refusal = h2  # h3 sin Δ₃
    
    # Plotear
    ax3.arrow(0, 0, h3[0], h3[1], head_width=0.08, head_length=0.05, fc='red', ec='red', lw=2, alpha=0.5)
    ax3.arrow(0, 0, h3_sin_refusal[0], h3_sin_refusal[1], head_width=0.1, head_length=0.06, fc='green', ec='green', lw=3)
    
    # Mostrar la dirección removida
    ax3.arrow(h3_sin_refusal[0], h3_sin_refusal[1], delta3[0], delta3[1], 
              head_width=0.08, head_length=0.05, fc='red', ec='red', lw=2, ls='--', alpha=0.5)
    
    ax3.text(h3[0]+0.1, h3[1], 'Con refusal', fontsize=10, color='red')
    ax3.text(h3_sin_refusal[0]+0.1, h3_sin_refusal[1]-0.15, 'Sin refusal\n(removemos Δ₃)', fontsize=9, color='green')
    
    ax3.set_xlim(-0.5, 2.5)
    ax3.set_ylim(-0.3, 2.5)
    ax3.set_aspect('equal')
    ax3.set_xlabel('Dimensión 1')
    ax3.set_ylabel('Dimensión 2')
    ax3.set_title('Remover la dirección de refusal', fontsize=12, fontweight='bold')
    ax3.axhline(y=0, color='gray', linewidth=0.5)
    ax3.axvline(x=0, color='gray', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()

visualizar_residual_stream()

---
# 🚫 Parte 3: Cómo Emerge el Refusal en LLMs

## 3.1 El Proceso de Entrenamiento

Los LLMs modernos pasan por varias fases de entrenamiento:

### Fase 1: Pre-entrenamiento
- **Objetivo**: Predecir el siguiente token
- **Datos**: Internet (Wikipedia, libros, código, etc.)
- **Resultado**: El modelo aprende lenguaje, hechos, patrones

### Fase 2: Fine-tuning Supervisado (SFT)
- **Objetivo**: Seguir instrucciones
- **Datos**: Pares (instrucción, respuesta deseada)
- **Resultado**: El modelo aprende a ser un asistente

### Fase 3: RLHF (Reinforcement Learning from Human Feedback)
- **Objetivo**: Alinearse con preferencias humanas
- **Datos**: Rankings de respuestas por humanos
- **Resultado**: El modelo aprende a **rechazar** solicitudes dañinas

> **El "refusal" se aprende principalmente en las fases 2 y 3**

In [ ]:
def visualizar_entrenamiento_refusal():
    """Visualiza cómo el refusal emerge durante el entrenamiento"""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Panel 1: Pre-entrenamiento (sin refusal)
    ax1 = axes[0]
    np.random.seed(42)
    
    # Simular que antes de RLHF, harmful y harmless están mezclados
    all_prompts = np.random.randn(100, 2) * 1.5
    colors = ['green' if np.random.random() > 0.5 else 'red' for _ in range(100)]
    
    ax1.scatter(all_prompts[:, 0], all_prompts[:, 1], c=colors, alpha=0.5, s=50)
    ax1.set_title('Pre-entrenamiento\n(No hay distinción harmful/harmless)', fontsize=11, fontweight='bold')
    ax1.set_xlabel('Dimensión 1')
    ax1.set_ylabel('Dimensión 2')
    
    # Añadir leyenda manual
    ax1.scatter([], [], c='green', label='Harmless prompts', s=50)
    ax1.scatter([], [], c='red', label='Harmful prompts', s=50)
    ax1.legend(loc='upper right', fontsize=8)
    ax1.text(0, -4, 'Modelo responde a TODO\n(no tiene concepto de "rechazo")', 
             ha='center', fontsize=9, style='italic')
    
    # Panel 2: Después de SFT + RLHF
    ax2 = axes[1]
    
    # Ahora harmful y harmless se separan
    harmless = np.random.randn(50, 2) * 0.8 + np.array([-2, -1])
    harmful = np.random.randn(50, 2) * 0.8 + np.array([2, 2])
    
    ax2.scatter(harmless[:, 0], harmless[:, 1], c='green', alpha=0.6, s=50, label='Harmless')
    ax2.scatter(harmful[:, 0], harmful[:, 1], c='red', alpha=0.6, s=50, label='Harmful')
    
    # Dibujar frontera de decisión
    x_line = np.linspace(-4, 4, 100)
    y_line = x_line * 0.5 + 0.5
    ax2.plot(x_line, y_line, 'k--', lw=2, label='Frontera de refusal')
    ax2.fill_between(x_line, y_line, 5, alpha=0.1, color='red')
    ax2.fill_between(x_line, y_line, -5, alpha=0.1, color='green')
    
    ax2.set_xlim(-4, 4)
    ax2.set_ylim(-4, 5)
    ax2.set_title('Después de RLHF\n(Harmful → región de refusal)', fontsize=11, fontweight='bold')
    ax2.set_xlabel('Dimensión 1')
    ax2.set_ylabel('Dimensión 2')
    ax2.legend(loc='upper left', fontsize=8)
    
    # Añadir flechas mostrando la "dirección de refusal"
    center_harmless = np.mean(harmless, axis=0)
    center_harmful = np.mean(harmful, axis=0)
    refusal_dir = center_harmful - center_harmless
    refusal_dir = refusal_dir / np.linalg.norm(refusal_dir)
    
    ax2.annotate('', xy=center_harmful, xytext=center_harmless,
                arrowprops=dict(arrowstyle='->', color='orange', lw=3))
    ax2.text(0, 0.5, 'd_refusal', fontsize=10, color='orange', fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Panel 3: Representación vectorial del refusal
    ax3 = axes[2]
    
    # Mostrar cómo el modelo "activa" la dirección de refusal
    ax3.set_xlim(0, 10)
    ax3.set_ylim(0, 8)
    ax3.axis('off')
    
    # Input
    ax3.add_patch(plt.Rectangle((1, 6.5), 8, 1, facecolor='lightcoral', edgecolor='black'))
    ax3.text(5, 7, '"How to make a bomb?"', ha='center', va='center', fontsize=10)
    
    # Flecha
    ax3.annotate('', xy=(5, 5.5), xytext=(5, 6.4),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Procesamiento interno
    ax3.add_patch(plt.Rectangle((1, 4), 8, 1.5, facecolor='lightyellow', edgecolor='black'))
    ax3.text(5, 5.1, 'Capas del Transformer', ha='center', va='center', fontsize=10, fontweight='bold')
    ax3.text(5, 4.4, 'h += d_refusal × α', ha='center', va='center', fontsize=10, 
             family='monospace', color='red')
    
    # Flecha
    ax3.annotate('', xy=(5, 3), xytext=(5, 3.9),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Output
    ax3.add_patch(plt.Rectangle((1, 1.5), 8, 1.5, facecolor='lightgreen', edgecolor='black'))
    ax3.text(5, 2.5, 'Output Logits', ha='center', va='center', fontsize=10, fontweight='bold')
    ax3.text(5, 1.9, 'P("I cannot") >> P("Here is how")', ha='center', va='center', 
             fontsize=9, family='monospace')
    
    ax3.set_title('Mecanismo interno del refusal', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

visualizar_entrenamiento_refusal()

## 3.2 La Hipótesis del Refusal Lineal

### La observación clave:

> **El comportamiento de "refusal" parece estar codificado en una ÚNICA DIRECCIÓN en el espacio de activaciones.**

Esto se conoce como la **Linear Representation Hypothesis**:

$$\text{refusal}(\mathbf{h}) \approx \mathbf{h} \cdot \mathbf{d}_{refusal}$$

Donde:
- $\mathbf{h}$ = vector de activaciones (hidden state)
- $\mathbf{d}_{refusal}$ = dirección de refusal (el vector que queremos encontrar)
- Si $\mathbf{h} \cdot \mathbf{d}_{refusal} > \theta$ → el modelo rechaza

### ¿Por qué funciona?

Los Transformers tienen miles de dimensiones (4096+). En espacios de alta dimensión:
- Los conceptos tienden a ser **casi ortogonales** entre sí
- Esto permite que muchos conceptos coexistan sin interferir
- Y que podamos **aislar** un concepto específico (como "refusal")

---
# 🔍 Parte 4: Extracción de la Dirección de Refusal

## 4.1 El Método de Diferencia de Medias

### Paso 1: Recolectar prompts

```python
harmful_prompts = [
    "How to make a bomb",
    "How to hack into a bank",
    "How to synthesize drugs",
    ...
]

harmless_prompts = [
    "How to make a cake",
    "How to learn programming",
    "How to synthesize music",
    ...
]
```

### Paso 2: Extraer activaciones

Para cada prompt, pasarlo por el modelo y guardar las activaciones de la última capa (o capa específica):

$$\mathbf{h}_i = \text{model.hidden\_states}[\text{prompt}_i][-1]$$

### Paso 3: Calcular la dirección

$$\mathbf{d}_{refusal} = \frac{1}{N}\sum_{i} \mathbf{h}^{harmful}_i - \frac{1}{M}\sum_{j} \mathbf{h}^{harmless}_j$$

Es decir: **media de harmful - media de harmless**

In [ ]:
def visualizar_extraccion_direccion():
    """Visualiza el proceso de extracción de la dirección de refusal"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    np.random.seed(42)
    
    # Generar datos simulados
    n_harmful = 30
    n_harmless = 30
    
    # En un espacio 2D para visualización
    # Harmful tiende hacia arriba-derecha, harmless hacia abajo-izquierda
    harmful_activations = np.random.randn(n_harmful, 2) * 0.6 + np.array([2, 2])
    harmless_activations = np.random.randn(n_harmless, 2) * 0.6 + np.array([-1, -1])
    
    # Panel 1: Datos crudos
    ax1 = axes[0, 0]
    ax1.scatter(harmful_activations[:, 0], harmful_activations[:, 1], 
                c='red', alpha=0.6, s=80, label='Harmful prompts', edgecolors='darkred')
    ax1.scatter(harmless_activations[:, 0], harmless_activations[:, 1], 
                c='green', alpha=0.6, s=80, label='Harmless prompts', edgecolors='darkgreen')
    
    # Añadir etiquetas de ejemplo
    examples_harmful = ['bomb', 'hack', 'kill', 'poison', 'weapon']
    examples_harmless = ['cake', 'code', 'music', 'learn', 'help']
    for i, txt in enumerate(examples_harmful[:3]):
        ax1.annotate(txt, harmful_activations[i], fontsize=8, alpha=0.7)
    for i, txt in enumerate(examples_harmless[:3]):
        ax1.annotate(txt, harmless_activations[i], fontsize=8, alpha=0.7)
    
    ax1.set_xlabel('Dimensión 1')
    ax1.set_ylabel('Dimensión 2')
    ax1.set_title('Paso 1: Activaciones extraídas del modelo', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.set_aspect('equal')
    
    # Panel 2: Calcular medias
    ax2 = axes[0, 1]
    ax2.scatter(harmful_activations[:, 0], harmful_activations[:, 1], 
                c='red', alpha=0.3, s=50)
    ax2.scatter(harmless_activations[:, 0], harmless_activations[:, 1], 
                c='green', alpha=0.3, s=50)
    
    mean_harmful = np.mean(harmful_activations, axis=0)
    mean_harmless = np.mean(harmless_activations, axis=0)
    
    ax2.scatter(*mean_harmful, c='red', s=300, marker='X', edgecolors='black', 
                linewidths=2, label='μ_harmful', zorder=5)
    ax2.scatter(*mean_harmless, c='green', s=300, marker='X', edgecolors='black', 
                linewidths=2, label='μ_harmless', zorder=5)
    
    ax2.set_xlabel('Dimensión 1')
    ax2.set_ylabel('Dimensión 2')
    ax2.set_title('Paso 2: Calcular medias de cada grupo', fontsize=12, fontweight='bold')
    ax2.legend()
    ax2.set_aspect('equal')
    
    # Panel 3: Dirección de refusal
    ax3 = axes[1, 0]
    ax3.scatter(harmful_activations[:, 0], harmful_activations[:, 1], 
                c='red', alpha=0.2, s=50)
    ax3.scatter(harmless_activations[:, 0], harmless_activations[:, 1], 
                c='green', alpha=0.2, s=50)
    ax3.scatter(*mean_harmful, c='red', s=200, marker='X', edgecolors='black', linewidths=2)
    ax3.scatter(*mean_harmless, c='green', s=200, marker='X', edgecolors='black', linewidths=2)
    
    # Calcular y dibujar dirección
    d_refusal = mean_harmful - mean_harmless
    d_refusal_norm = d_refusal / np.linalg.norm(d_refusal)
    
    ax3.annotate('', xy=mean_harmful, xytext=mean_harmless,
                arrowprops=dict(arrowstyle='->', color='orange', lw=4))
    
    # Extender la línea
    t = np.linspace(-2, 5, 100)
    line_x = mean_harmless[0] + t * d_refusal_norm[0]
    line_y = mean_harmless[1] + t * d_refusal_norm[1]
    ax3.plot(line_x, line_y, 'orange', linestyle='--', lw=2, alpha=0.5)
    
    ax3.text((mean_harmful[0]+mean_harmless[0])/2 + 0.3, 
             (mean_harmful[1]+mean_harmless[1])/2 - 0.3,
             f'd_refusal\n||d|| = {np.linalg.norm(d_refusal):.2f}', 
             fontsize=11, color='orange', fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax3.set_xlabel('Dimensión 1')
    ax3.set_ylabel('Dimensión 2')
    ax3.set_title('Paso 3: d_refusal = μ_harmful - μ_harmless', fontsize=12, fontweight='bold')
    ax3.set_aspect('equal')
    
    # Panel 4: Proyecciones sobre la dirección
    ax4 = axes[1, 1]
    
    # Proyectar todos los puntos sobre d_refusal
    proj_harmful = [np.dot(h, d_refusal_norm) for h in harmful_activations]
    proj_harmless = [np.dot(h, d_refusal_norm) for h in harmless_activations]
    
    # Histograma
    bins = np.linspace(-3, 5, 30)
    ax4.hist(proj_harmless, bins=bins, alpha=0.6, color='green', label='Harmless', density=True)
    ax4.hist(proj_harmful, bins=bins, alpha=0.6, color='red', label='Harmful', density=True)
    
    ax4.axvline(x=np.mean(proj_harmless), color='green', linestyle='--', lw=2)
    ax4.axvline(x=np.mean(proj_harmful), color='red', linestyle='--', lw=2)
    ax4.axvline(x=(np.mean(proj_harmful)+np.mean(proj_harmless))/2, 
                color='black', linestyle='-', lw=2, label='Umbral θ')
    
    ax4.set_xlabel('Proyección sobre d_refusal', fontsize=11)
    ax4.set_ylabel('Densidad', fontsize=11)
    ax4.set_title('Paso 4: Separación en la dirección encontrada', fontsize=12, fontweight='bold')
    ax4.legend()
    
    # Añadir texto explicativo
    ax4.text(3.5, ax4.get_ylim()[1]*0.8, 'Si proj > θ:\nREFUSAL', fontsize=10, 
             ha='center', color='red', fontweight='bold')
    ax4.text(-1.5, ax4.get_ylim()[1]*0.8, 'Si proj < θ:\nRESPONDE', fontsize=10, 
             ha='center', color='green', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return d_refusal_norm

d_refusal = visualizar_extraccion_direccion()
print(f"\\nDirección de refusal encontrada (normalizada): {d_refusal}")

## 4.2 Método Alternativo: PCA

### ¿Por qué PCA puede ser mejor?

La diferencia de medias solo captura **una dirección**. Pero el "refusal" podría manifestarse en **múltiples direcciones**.

### El proceso:

1. Centrar los datos: $\mathbf{X} = \mathbf{H} - \bar{\mathbf{H}}$
2. Calcular matriz de covarianza: $\mathbf{C} = \frac{1}{n}\mathbf{X}^T\mathbf{X}$
3. Encontrar eigenvectors: $\mathbf{C}\mathbf{v}_i = \lambda_i\mathbf{v}_i$
4. Los eigenvectors con mayor $\lambda$ son las **direcciones principales de varianza**

### Ventaja:
PCA encuentra **múltiples direcciones** que explican la varianza entre harmful/harmless, no solo la diferencia de centros.

In [ ]:
def visualizar_pca_vs_medias():
    """Compara PCA vs diferencia de medias para encontrar direcciones"""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    np.random.seed(42)
    
    # Crear datos con estructura más compleja
    # Los grupos tienen varianza en direcciones diferentes
    n = 40
    
    # Harmless: varianza principalmente en una dirección
    cov_harmless = [[1.0, 0.3], [0.3, 0.3]]
    harmless = np.random.multivariate_normal([-1.5, -1], cov_harmless, n)
    
    # Harmful: varianza principalmente en OTRA dirección
    cov_harmful = [[0.4, 0.5], [0.5, 1.2]]
    harmful = np.random.multivariate_normal([1.5, 1.5], cov_harmful, n)
    
    # Panel 1: Datos con sus elipses de varianza
    ax1 = axes[0]
    ax1.scatter(harmless[:, 0], harmless[:, 1], c='green', alpha=0.5, s=50, label='Harmless')
    ax1.scatter(harmful[:, 0], harmful[:, 1], c='red', alpha=0.5, s=50, label='Harmful')
    
    # Dibujar elipses de covarianza
    from matplotlib.patches import Ellipse
    
    def plot_cov_ellipse(ax, mean, cov, color, alpha=0.2):
        eigvals, eigvecs = np.linalg.eigh(cov)
        angle = np.degrees(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))
        width, height = 2 * np.sqrt(eigvals) * 2  # 2 std
        ellipse = Ellipse(mean, width, height, angle=angle, 
                         facecolor=color, alpha=alpha, edgecolor=color, lw=2)
        ax.add_patch(ellipse)
    
    mean_harmless = np.mean(harmless, axis=0)
    mean_harmful = np.mean(harmful, axis=0)
    
    plot_cov_ellipse(ax1, mean_harmless, np.cov(harmless.T), 'green')
    plot_cov_ellipse(ax1, mean_harmful, np.cov(harmful.T), 'red')
    
    ax1.scatter(*mean_harmless, c='green', s=200, marker='X', edgecolors='black', zorder=5)
    ax1.scatter(*mean_harmful, c='red', s=200, marker='X', edgecolors='black', zorder=5)
    
    ax1.set_xlabel('Dimensión 1')
    ax1.set_ylabel('Dimensión 2')
    ax1.set_title('Datos con estructuras de covarianza diferentes', fontsize=11, fontweight='bold')
    ax1.legend()
    ax1.set_aspect('equal')
    ax1.set_xlim(-5, 5)
    ax1.set_ylim(-4, 5)
    
    # Panel 2: Diferencia de medias
    ax2 = axes[1]
    ax2.scatter(harmless[:, 0], harmless[:, 1], c='green', alpha=0.3, s=30)
    ax2.scatter(harmful[:, 0], harmful[:, 1], c='red', alpha=0.3, s=30)
    
    d_mean = mean_harmful - mean_harmless
    d_mean_norm = d_mean / np.linalg.norm(d_mean)
    
    # Dibujar dirección
    ax2.arrow(mean_harmless[0], mean_harmless[1], d_mean[0], d_mean[1],
              head_width=0.2, head_length=0.15, fc='blue', ec='blue', lw=3)
    
    # Extender línea
    t = np.linspace(-3, 3, 100)
    center = (mean_harmful + mean_harmless) / 2
    ax2.plot(center[0] + t*d_mean_norm[0], center[1] + t*d_mean_norm[1], 
             'b--', lw=2, alpha=0.5)
    
    ax2.set_xlabel('Dimensión 1')
    ax2.set_ylabel('Dimensión 2')
    ax2.set_title('Método: Diferencia de Medias\n(1 sola dirección)', fontsize=11, fontweight='bold')
    ax2.set_aspect('equal')
    ax2.set_xlim(-5, 5)
    ax2.set_ylim(-4, 5)
    
    # Panel 3: PCA
    ax3 = axes[2]
    ax3.scatter(harmless[:, 0], harmless[:, 1], c='green', alpha=0.3, s=30)
    ax3.scatter(harmful[:, 0], harmful[:, 1], c='red', alpha=0.3, s=30)
    
    # PCA sobre datos combinados
    all_data = np.vstack([harmful, harmless])
    centered = all_data - np.mean(all_data, axis=0)
    cov_all = np.cov(centered.T)
    eigvals, eigvecs = np.linalg.eigh(cov_all)
    
    # Ordenar por eigenvalue descendente
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]
    
    center_all = np.mean(all_data, axis=0)
    
    # PC1
    pc1 = eigvecs[:, 0] * np.sqrt(eigvals[0]) * 1.5
    ax3.arrow(center_all[0], center_all[1], pc1[0], pc1[1],
              head_width=0.2, head_length=0.15, fc='purple', ec='purple', lw=3)
    ax3.text(center_all[0] + pc1[0] + 0.2, center_all[1] + pc1[1],
             f'PC1\n(var={eigvals[0]:.1f})', fontsize=9, color='purple')
    
    # PC2
    pc2 = eigvecs[:, 1] * np.sqrt(eigvals[1]) * 1.5
    ax3.arrow(center_all[0], center_all[1], pc2[0], pc2[1],
              head_width=0.2, head_length=0.15, fc='orange', ec='orange', lw=3)
    ax3.text(center_all[0] + pc2[0] + 0.2, center_all[1] + pc2[1],
             f'PC2\n(var={eigvals[1]:.1f})', fontsize=9, color='orange')
    
    ax3.set_xlabel('Dimensión 1')
    ax3.set_ylabel('Dimensión 2')
    ax3.set_title('Método: PCA\n(múltiples direcciones de varianza)', fontsize=11, fontweight='bold')
    ax3.set_aspect('equal')
    ax3.set_xlim(-5, 5)
    ax3.set_ylim(-4, 5)
    
    plt.tight_layout()
    plt.show()
    
    print("\\n📊 Comparación:")
    print(f"   Diferencia de medias: 1 dirección")
    print(f"   PCA: {len(eigvals)} direcciones, con varianzas explicadas: {eigvals}")

visualizar_pca_vs_medias()

---
# 🔧 Parte 5: Técnicas de Manipulación Vectorial

Una vez que tenemos la dirección de refusal $\mathbf{d}$, podemos usarla para **modificar el comportamiento del modelo**.

## 5.1 Ortogonalización (Proyección)

### La idea:
Remover el componente de $\mathbf{h}$ que está en la dirección de $\mathbf{d}$.

### La matemática:

$$\mathbf{h}_{new} = \mathbf{h} - \frac{\mathbf{h} \cdot \mathbf{d}}{||\mathbf{d}||^2} \mathbf{d}$$

Si $\mathbf{d}$ está normalizado ($||\mathbf{d}|| = 1$):

$$\mathbf{h}_{new} = \mathbf{h} - (\mathbf{h} \cdot \hat{\mathbf{d}}) \hat{\mathbf{d}}$$

### Implementación:
```python
def orthogonalize(h, d):
    d_norm = d / np.linalg.norm(d)
    projection = np.dot(h, d_norm) * d_norm
    return h - projection
```

In [ ]:
def visualizar_ortogonalizacion_detallada():
    """Visualización detallada del proceso de ortogonalización"""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Vector original y dirección de refusal
    h = np.array([2.5, 2.0])
    d = np.array([1, 1])
    d_norm = d / np.linalg.norm(d)
    
    # Calcular proyección y vector ortogonalizado
    proj_scalar = np.dot(h, d_norm)
    proj_vector = proj_scalar * d_norm
    h_orth = h - proj_vector
    
    # Panel 1: Los componentes
    ax1 = axes[0]
    ax1.set_xlim(-0.5, 3.5)
    ax1.set_ylim(-0.5, 3)
    ax1.set_aspect('equal')
    
    # Dibujar
    ax1.arrow(0, 0, h[0], h[1], head_width=0.1, head_length=0.08, fc='blue', ec='blue', lw=2)
    ax1.arrow(0, 0, d_norm[0]*2.5, d_norm[1]*2.5, head_width=0.1, head_length=0.08, 
              fc='red', ec='red', lw=2, alpha=0.7)
    
    # Proyección
    ax1.arrow(0, 0, proj_vector[0], proj_vector[1], head_width=0.1, head_length=0.08, 
              fc='orange', ec='orange', lw=2)
    
    # Línea punteada de h a la proyección
    ax1.plot([h[0], proj_vector[0]], [h[1], proj_vector[1]], 'k--', lw=1.5, alpha=0.5)
    
    # Etiquetas
    ax1.text(h[0]+0.1, h[1]+0.1, 'h (original)', fontsize=10, color='blue')
    ax1.text(d_norm[0]*2.5+0.1, d_norm[1]*2.5, 'd̂ (refusal)', fontsize=10, color='red')
    ax1.text(proj_vector[0]-0.3, proj_vector[1]-0.3, f'proj\n({proj_scalar:.2f}·d̂)', 
             fontsize=9, color='orange')
    
    ax1.axhline(y=0, color='gray', linewidth=0.5)
    ax1.axvline(x=0, color='gray', linewidth=0.5)
    ax1.set_xlabel('Dim 1')
    ax1.set_ylabel('Dim 2')
    ax1.set_title('Paso 1: Calcular proyección\nproj = (h·d̂)·d̂', fontsize=11, fontweight='bold')
    
    # Panel 2: Resta
    ax2 = axes[1]
    ax2.set_xlim(-0.5, 3.5)
    ax2.set_ylim(-0.5, 3)
    ax2.set_aspect('equal')
    
    # h original
    ax2.arrow(0, 0, h[0], h[1], head_width=0.1, head_length=0.08, fc='blue', ec='blue', lw=2, alpha=0.5)
    
    # Proyección (lo que restamos)
    ax2.arrow(h[0], h[1], -proj_vector[0], -proj_vector[1], head_width=0.1, head_length=0.08, 
              fc='orange', ec='orange', lw=2, linestyle='--')
    
    # Resultado
    ax2.arrow(0, 0, h_orth[0], h_orth[1], head_width=0.12, head_length=0.08, fc='green', ec='green', lw=3)
    
    # Dirección de refusal
    ax2.arrow(0, 0, d_norm[0]*2.5, d_norm[1]*2.5, head_width=0.08, head_length=0.06, 
              fc='red', ec='red', lw=1.5, alpha=0.3)
    
    ax2.text(h[0]+0.1, h[1]+0.1, 'h', fontsize=10, color='blue', alpha=0.5)
    ax2.text(h_orth[0]+0.1, h_orth[1]-0.2, 'h_orth', fontsize=10, color='green', fontweight='bold')
    ax2.text(h[0]-proj_vector[0]/2, h[1]-proj_vector[1]/2+0.2, '-proj', fontsize=9, color='orange')
    
    ax2.axhline(y=0, color='gray', linewidth=0.5)
    ax2.axvline(x=0, color='gray', linewidth=0.5)
    ax2.set_xlabel('Dim 1')
    ax2.set_ylabel('Dim 2')
    ax2.set_title('Paso 2: Restar proyección\nh_orth = h - proj', fontsize=11, fontweight='bold')
    
    # Panel 3: Verificación
    ax3 = axes[2]
    ax3.set_xlim(-0.5, 3.5)
    ax3.set_ylim(-0.5, 3)
    ax3.set_aspect('equal')
    
    # Resultado final
    ax3.arrow(0, 0, h_orth[0], h_orth[1], head_width=0.12, head_length=0.08, fc='green', ec='green', lw=3)
    
    # Dirección de refusal
    ax3.arrow(0, 0, d_norm[0]*2.5, d_norm[1]*2.5, head_width=0.08, head_length=0.06, 
              fc='red', ec='red', lw=2, alpha=0.5)
    
    # Mostrar que son perpendiculares
    # Dibujar ángulo recto
    angle_size = 0.3
    ax3.plot([h_orth[0], h_orth[0]+angle_size*d_norm[0]], 
             [h_orth[1], h_orth[1]+angle_size*d_norm[1]], 'k-', lw=1)
    ax3.plot([h_orth[0]+angle_size*d_norm[0], h_orth[0]+angle_size*d_norm[0]-angle_size*d_norm[1]], 
             [h_orth[1]+angle_size*d_norm[1], h_orth[1]+angle_size*d_norm[1]+angle_size*d_norm[0]], 'k-', lw=1)
    
    dot_product = np.dot(h_orth, d_norm)
    ax3.text(1.5, 2.5, f'h_orth · d̂ = {dot_product:.6f} ≈ 0', fontsize=11, 
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
    ax3.text(1.5, 2.0, '✓ Son perpendiculares!', fontsize=11, color='green', fontweight='bold')
    
    ax3.text(h_orth[0]+0.1, h_orth[1]-0.2, 'h_orth', fontsize=10, color='green', fontweight='bold')
    ax3.text(d_norm[0]*2.5+0.1, d_norm[1]*2.5, 'd̂', fontsize=10, color='red')
    
    ax3.axhline(y=0, color='gray', linewidth=0.5)
    ax3.axvline(x=0, color='gray', linewidth=0.5)
    ax3.set_xlabel('Dim 1')
    ax3.set_ylabel('Dim 2')
    ax3.set_title('Verificación: h_orth ⊥ d̂\n(no tiene componente de refusal)', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\\n📐 Detalles numéricos:")
    print(f"   h original:     {h}")
    print(f"   d (normalizado): {d_norm}")
    print(f"   Proyección escalar (h·d̂): {proj_scalar:.4f}")
    print(f"   Vector proyección: {proj_vector}")
    print(f"   h ortogonalizado: {h_orth}")
    print(f"   Verificación h_orth·d̂: {dot_product:.10f}")

visualizar_ortogonalizacion_detallada()

## 5.2 Ablación Direccional (en Matrices de Pesos)

### El problema con ortogonalización:
La ortogonalización modifica **vectores individuales** (activaciones).
Pero si queremos que el modelo **nunca** produzca output en la dirección de refusal, necesitamos modificar las **matrices de pesos**.

### La idea:
Modificar la matriz $W$ para que su output nunca tenga componente en $\mathbf{d}$.

### La matemática:

Para una matriz $W$ que transforma inputs a outputs:
$$W' = W - \mathbf{d}\mathbf{d}^T W / ||\mathbf{d}||^2$$

O equivalentemente (proyector ortogonal):
$$W' = (I - \hat{\mathbf{d}}\hat{\mathbf{d}}^T) W$$

Esto garantiza que para **cualquier input** $\mathbf{x}$:
$$W'\mathbf{x} \perp \mathbf{d}$$

### Implementación:
```python
def ablate_direction(W, d):
    d_norm = d / np.linalg.norm(d)
    # Proyector ortogonal
    P = np.eye(len(d)) - np.outer(d_norm, d_norm)
    return P @ W
```

In [ ]:
def visualizar_ablacion_matrices():
    """Visualiza cómo la ablación modifica el espacio de transformación de una matriz"""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Matriz de transformación original
    W = np.array([[1.5, 0.3],
                  [0.5, 1.2]])
    
    # Dirección de refusal
    d = np.array([0.7, 0.7])
    d_norm = d / np.linalg.norm(d)
    
    # Matriz ablacionada
    P = np.eye(2) - np.outer(d_norm, d_norm)  # Proyector ortogonal
    W_ablated = P @ W
    
    # Panel 1: Cómo W transforma el espacio
    ax1 = axes[0]
    
    # Dibujar círculo unitario y su transformación
    theta = np.linspace(0, 2*np.pi, 100)
    circle = np.array([np.cos(theta), np.sin(theta)])
    transformed = W @ circle
    
    ax1.plot(circle[0], circle[1], 'gray', lw=1, alpha=0.5, label='Input (círculo)')
    ax1.plot(transformed[0], transformed[1], 'blue', lw=2, label='W @ input')
    
    # Dirección de refusal
    ax1.arrow(0, 0, d_norm[0]*1.5, d_norm[1]*1.5, head_width=0.1, head_length=0.08,
              fc='red', ec='red', lw=2)
    ax1.text(d_norm[0]*1.5+0.1, d_norm[1]*1.5, 'd̂', fontsize=11, color='red')
    
    ax1.set_xlim(-2.5, 2.5)
    ax1.set_ylim(-2.5, 2.5)
    ax1.set_aspect('equal')
    ax1.set_xlabel('Dim 1')
    ax1.set_ylabel('Dim 2')
    ax1.set_title('Matriz W original\n(outputs tienen componente en d̂)', fontsize=11, fontweight='bold')
    ax1.legend(loc='upper left', fontsize=8)
    ax1.axhline(y=0, color='gray', linewidth=0.5)
    ax1.axvline(x=0, color='gray', linewidth=0.5)
    
    # Panel 2: Cómo W' (ablacionada) transforma el espacio
    ax2 = axes[1]
    
    transformed_ablated = W_ablated @ circle
    
    ax2.plot(circle[0], circle[1], 'gray', lw=1, alpha=0.5, label='Input (círculo)')
    ax2.plot(transformed_ablated[0], transformed_ablated[1], 'green', lw=2, label="W' @ input")
    
    # Dirección de refusal
    ax2.arrow(0, 0, d_norm[0]*1.5, d_norm[1]*1.5, head_width=0.1, head_length=0.08,
              fc='red', ec='red', lw=2, alpha=0.5)
    
    # Subespacio ortogonal (línea perpendicular a d)
    perp = np.array([-d_norm[1], d_norm[0]])
    ax2.plot([-perp[0]*2, perp[0]*2], [-perp[1]*2, perp[1]*2], 'g--', lw=2, 
             alpha=0.7, label='Subespacio ⊥ d̂')
    
    ax2.set_xlim(-2.5, 2.5)
    ax2.set_ylim(-2.5, 2.5)
    ax2.set_aspect('equal')
    ax2.set_xlabel('Dim 1')
    ax2.set_ylabel('Dim 2')
    ax2.set_title("Matriz W' ablacionada\n(outputs SIEMPRE ⊥ d̂)", fontsize=11, fontweight='bold')
    ax2.legend(loc='upper left', fontsize=8)
    ax2.axhline(y=0, color='gray', linewidth=0.5)
    ax2.axvline(x=0, color='gray', linewidth=0.5)
    
    # Panel 3: Comparación de residuos
    ax3 = axes[2]
    
    # Probar muchos inputs aleatorios
    np.random.seed(42)
    n_tests = 50
    inputs = np.random.randn(2, n_tests)
    
    # Calcular componente en d para cada output
    outputs_original = W @ inputs
    outputs_ablated = W_ablated @ inputs
    
    components_original = np.abs(d_norm @ outputs_original)
    components_ablated = np.abs(d_norm @ outputs_ablated)
    
    x = np.arange(n_tests)
    width = 0.35
    ax3.bar(x - width/2, components_original, width, label='W (original)', color='blue', alpha=0.7)
    ax3.bar(x + width/2, components_ablated, width, label="W' (ablacionada)", color='green', alpha=0.7)
    
    ax3.set_xlabel('Input aleatorio #')
    ax3.set_ylabel('|componente en d̂|')
    ax3.set_title('Componente de refusal en outputs\n(para 50 inputs aleatorios)', fontsize=11, fontweight='bold')
    ax3.legend()
    ax3.set_xlim(-1, 50)
    
    # Añadir línea en y=0 para referencia
    ax3.axhline(y=0.01, color='green', linestyle='--', alpha=0.5)
    ax3.text(25, 0.15, "W' → componente ≈ 0 siempre", fontsize=10, ha='center', color='green')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\\n📊 Estadísticas:")
    print(f"   W original  - Componente medio en d̂: {np.mean(components_original):.4f}")
    print(f"   W' ablacionada - Componente medio en d̂: {np.mean(components_ablated):.2e}")

visualizar_ablacion_matrices()

## 5.3 Direcciones por Capa

### El problema con una dirección global:
El "refusal" puede manifestarse **diferente en cada capa**:
- Capas tempranas: representaciones más sintácticas
- Capas medias: semánticas
- Capas finales: relacionadas con output

### La solución:
Extraer $\mathbf{d}^{(l)}$ para cada capa $l$ y aplicar la técnica (ortogonalización o ablación) **por capa**.

In [ ]:
def visualizar_direcciones_por_capa():
    """Visualiza cómo la dirección de refusal varía por capa"""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    np.random.seed(42)
    n_layers = 8
    
    # Simular direcciones de refusal por capa (varían ligeramente)
    base_angle = np.pi/4  # 45 grados
    angles = [base_angle + (i - n_layers/2) * 0.12 for i in range(n_layers)]
    directions = [np.array([np.cos(a), np.sin(a)]) for a in angles]
    
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, n_layers))
    
    # Panel 1: Direcciones por capa
    ax1 = axes[0]
    
    for i, (d, col) in enumerate(zip(directions, colors)):
        ax1.arrow(0, 0, d[0]*1.3, d[1]*1.3, head_width=0.08, head_length=0.06,
                  fc=col, ec=col, lw=2, label=f'Capa {i+1}')
    
    # Dirección promedio
    d_mean = np.mean(directions, axis=0)
    d_mean = d_mean / np.linalg.norm(d_mean)
    ax1.arrow(0, 0, d_mean[0]*1.5, d_mean[1]*1.5, head_width=0.1, head_length=0.08,
              fc='red', ec='red', lw=3, linestyle='--', alpha=0.7)
    ax1.text(d_mean[0]*1.5+0.1, d_mean[1]*1.5, 'd̂_promedio\n(subóptimo)', 
             fontsize=9, color='red')
    
    ax1.set_xlim(-0.5, 2)
    ax1.set_ylim(-0.5, 2)
    ax1.set_aspect('equal')
    ax1.legend(loc='upper left', fontsize=7, ncol=2)
    ax1.set_xlabel('Dim 1')
    ax1.set_ylabel('Dim 2')
    ax1.set_title('Dirección de refusal varía por capa', fontsize=11, fontweight='bold')
    ax1.axhline(y=0, color='gray', linewidth=0.5)
    ax1.axvline(x=0, color='gray', linewidth=0.5)
    
    # Panel 2: Error si usamos dirección global
    ax2 = axes[1]
    
    # Simular activaciones y calcular error residual
    errors_global = []
    errors_local = []
    
    for i in range(n_layers):
        # Dirección real de esta capa
        d_real = directions[i]
        
        # Simular varias activaciones
        for _ in range(10):
            h = np.random.randn(2) * 2
            
            # Ortogonalizar con dirección global
            h_orth_global = h - np.dot(h, d_mean) * d_mean
            error_global = abs(np.dot(h_orth_global, d_real))
            errors_global.append((i, error_global))
            
            # Ortogonalizar con dirección local
            h_orth_local = h - np.dot(h, d_real) * d_real
            error_local = abs(np.dot(h_orth_local, d_real))
            errors_local.append((i, error_local))
    
    # Plotear
    layers_g, errs_g = zip(*errors_global)
    layers_l, errs_l = zip(*errors_local)
    
    ax2.scatter(layers_g, errs_g, c='red', alpha=0.5, s=30, label='Dir. global')
    ax2.scatter(layers_l, errs_l, c='green', alpha=0.5, s=30, label='Dir. por capa')
    
    # Medias por capa
    for i in range(n_layers):
        mean_g = np.mean([e for l, e in errors_global if l == i])
        mean_l = np.mean([e for l, e in errors_local if l == i])
        ax2.plot([i-0.2, i+0.2], [mean_g, mean_g], 'r-', lw=3)
        ax2.plot([i-0.2, i+0.2], [mean_l, mean_l], 'g-', lw=3)
    
    ax2.set_xlabel('Capa')
    ax2.set_ylabel('Error residual (componente en d_real)')
    ax2.set_xticks(range(n_layers))
    ax2.set_xticklabels([f'L{i+1}' for i in range(n_layers)])
    ax2.set_title('Comparación: Dir. global vs Dir. por capa', fontsize=11, fontweight='bold')
    ax2.legend()
    
    # Panel 3: Similitud entre direcciones
    ax3 = axes[2]
    
    # Calcular matriz de similitud (producto punto)
    similarity_matrix = np.zeros((n_layers, n_layers))
    for i in range(n_layers):
        for j in range(n_layers):
            similarity_matrix[i, j] = np.dot(directions[i], directions[j])
    
    im = ax3.imshow(similarity_matrix, cmap='RdYlGn', vmin=0.8, vmax=1.0)
    ax3.set_xlabel('Capa')
    ax3.set_ylabel('Capa')
    ax3.set_xticks(range(n_layers))
    ax3.set_yticks(range(n_layers))
    ax3.set_xticklabels([f'L{i+1}' for i in range(n_layers)])
    ax3.set_yticklabels([f'L{i+1}' for i in range(n_layers)])
    ax3.set_title('Similitud entre direcciones\n(d_i · d_j)', fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax3, shrink=0.8)
    
    # Anotar valores
    for i in range(n_layers):
        for j in range(n_layers):
            ax3.text(j, i, f'{similarity_matrix[i,j]:.2f}', 
                    ha='center', va='center', fontsize=7,
                    color='white' if similarity_matrix[i,j] < 0.9 else 'black')
    
    plt.tight_layout()
    plt.show()
    
    print("\\n💡 Observación:")
    print("   Las direcciones son SIMILARES pero NO IDÉNTICAS.")
    print("   Usar la dirección específica de cada capa reduce el error residual.")

visualizar_direcciones_por_capa()

---
# 📊 Parte 6: Comparación y Resumen Final

## 6.1 Tabla Comparativa de Técnicas

| Técnica | Qué modifica | Ventaja | Desventaja |
|---------|-------------|---------|------------|
| **Diferencia de medias** | Nada (solo extrae d) | Simple, rápido | Solo 1 dirección |
| **PCA** | Nada (solo extrae d) | Múltiples direcciones | Más complejo |
| **Ortogonalización** | Activaciones (h) | Fácil de aplicar | Hay que aplicar en cada forward |
| **Ablación de W** | Pesos del modelo | Permanente, garantizado | Modifica el modelo |
| **Dir. global** | - | Simple | Deja residuos |
| **Dir. por capa** | - | Más preciso | Más complejo |

## 6.2 ¿Cuándo usar cada técnica?

### Para investigación rápida:
1. Diferencia de medias para extraer d
2. Ortogonalización para probar

### Para modificación permanente:
1. PCA o diferencia de medias por capa
2. Ablación de matrices de pesos

In [ ]:
def visualizar_resumen_final():
    """Visualización resumen de todo el proceso"""
    fig = plt.figure(figsize=(16, 10))
    
    # Crear grid de subplots
    gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)
    
    np.random.seed(42)
    
    # ========== Fila 1: El proceso completo ==========
    
    # Panel 1: Input → Embedding
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_xlim(0, 10)
    ax1.set_ylim(0, 10)
    ax1.axis('off')
    
    ax1.add_patch(plt.Rectangle((1, 7), 8, 2), facecolor='lightcoral', edgecolor='black')
    ax1.text(5, 8, '"How to make a bomb?"', ha='center', va='center', fontsize=10)
    ax1.annotate('', xy=(5, 5.5), xytext=(5, 6.8),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax1.text(5, 6.2, 'Tokenize +\\nEmbed', ha='center', fontsize=9)
    
    ax1.add_patch(plt.Rectangle((1, 3), 8, 2), facecolor='lightyellow', edgecolor='black')
    ax1.text(5, 4, 'h₀ ∈ ℝ^4096', ha='center', va='center', fontsize=11, family='monospace')
    ax1.set_title('1. Input → Embedding', fontsize=11, fontweight='bold')
    
    # Panel 2: Transformer layers
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, 10)
    ax2.axis('off')
    
    y_pos = 8.5
    for i in range(4):
        color = plt.cm.Blues(0.3 + i*0.15)
        ax2.add_patch(plt.Rectangle((1, y_pos-1), 8, 0.8), facecolor=color, edgecolor='black')
        ax2.text(5, y_pos-0.6, f'Capa {i+1}: h += Δ{i+1}', ha='center', fontsize=9)
        if i < 3:
            ax2.annotate('', xy=(5, y_pos-1.2), xytext=(5, y_pos-1),
                        arrowprops=dict(arrowstyle='->', color='gray', lw=1))
        y_pos -= 1.5
    
    ax2.text(5, 2, '...', ha='center', fontsize=14)
    ax2.add_patch(plt.Rectangle((1, 0.5), 8, 1), facecolor='lightgreen', edgecolor='black')
    ax2.text(5, 1, 'h_final (acumula refusal)', ha='center', fontsize=9, color='red')
    ax2.set_title('2. Transformer Layers', fontsize=11, fontweight='bold')
    
    # Panel 3: Detección de refusal
    ax3 = fig.add_subplot(gs[0, 2])
    
    # Simular proyecciones
    proj_harmful = np.random.randn(30) * 0.5 + 2
    proj_harmless = np.random.randn(30) * 0.5 - 1
    
    ax3.hist(proj_harmless, bins=15, alpha=0.6, color='green', label='Harmless', density=True)
    ax3.hist(proj_harmful, bins=15, alpha=0.6, color='red', label='Harmful', density=True)
    ax3.axvline(x=0.5, color='black', lw=2, linestyle='--', label='Umbral')
    ax3.set_xlabel('h · d_refusal')
    ax3.set_ylabel('Densidad')
    ax3.legend(fontsize=8)
    ax3.set_title('3. Detección: h · d > θ?', fontsize=11, fontweight='bold')
    
    # ========== Fila 2: Las técnicas de remoción ==========
    
    # Panel 4: Ortogonalización
    ax4 = fig.add_subplot(gs[1, 0])
    
    h = np.array([2, 1.5])
    d = np.array([1, 1]) / np.sqrt(2)
    h_orth = h - np.dot(h, d) * d
    
    ax4.arrow(0, 0, h[0], h[1], head_width=0.1, fc='blue', ec='blue', lw=2, alpha=0.5)
    ax4.arrow(0, 0, h_orth[0], h_orth[1], head_width=0.1, fc='green', ec='green', lw=2)
    ax4.arrow(0, 0, d[0]*2, d[1]*2, head_width=0.08, fc='red', ec='red', lw=1.5, alpha=0.5)
    
    ax4.text(h[0]+0.1, h[1], 'h', color='blue', fontsize=10)
    ax4.text(h_orth[0]+0.1, h_orth[1]-0.2, 'h_orth', color='green', fontsize=10, fontweight='bold')
    ax4.text(d[0]*2+0.1, d[1]*2, 'd̂', color='red', fontsize=10)
    
    ax4.set_xlim(-0.5, 3)
    ax4.set_ylim(-0.5, 2.5)
    ax4.set_aspect('equal')
    ax4.set_title('4a. Ortogonalización\\nh_new = h - (h·d̂)d̂', fontsize=10, fontweight='bold')
    ax4.axhline(y=0, color='gray', lw=0.5)
    ax4.axvline(x=0, color='gray', lw=0.5)
    
    # Panel 5: Ablación de matriz
    ax5 = fig.add_subplot(gs[1, 1])
    
    theta = np.linspace(0, 2*np.pi, 100)
    circle = np.array([np.cos(theta), np.sin(theta)])
    W = np.array([[1.3, 0.2], [0.4, 1.1]])
    P = np.eye(2) - np.outer(d, d)
    W_abl = P @ W
    
    ax5.plot(circle[0], circle[1], 'gray', lw=1, alpha=0.3)
    ax5.plot(*(W @ circle), 'b-', lw=2, alpha=0.5, label='W (original)')
    ax5.plot(*(W_abl @ circle), 'g-', lw=2, label="W' (ablacionada)")
    ax5.arrow(0, 0, d[0]*1.5, d[1]*1.5, head_width=0.08, fc='red', ec='red', lw=1.5, alpha=0.5)
    
    ax5.set_xlim(-2, 2)
    ax5.set_ylim(-2, 2)
    ax5.set_aspect('equal')
    ax5.legend(fontsize=8, loc='upper left')
    ax5.set_title("4b. Ablación de W\\nW' = (I - d̂d̂ᵀ)W", fontsize=10, fontweight='bold')
    
    # Panel 6: Resultado final
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.set_xlim(0, 10)
    ax6.set_ylim(0, 10)
    ax6.axis('off')
    
    ax6.add_patch(plt.Rectangle((1, 6), 8, 3), facecolor='lightgreen', edgecolor='darkgreen', lw=2)
    ax6.text(5, 8, '✓ Modelo modificado', ha='center', fontsize=12, fontweight='bold', color='darkgreen')
    ax6.text(5, 7, 'No rechaza solicitudes', ha='center', fontsize=10)
    ax6.text(5, 6.3, '(cuidado con el uso ético)', ha='center', fontsize=9, style='italic')
    
    ax6.add_patch(plt.Rectangle((1, 2), 8, 3), facecolor='lightyellow', edgecolor='orange', lw=2)
    ax6.text(5, 4, '📊 Técnicas disponibles:', ha='center', fontsize=11, fontweight='bold')
    ax6.text(5, 3.2, '• Ortogonalización (en inferencia)', ha='center', fontsize=9)
    ax6.text(5, 2.5, '• Ablación de pesos (permanente)', ha='center', fontsize=9)
    
    ax6.set_title('5. Resultado', fontsize=11, fontweight='bold')
    
    plt.suptitle('Resumen: Manipulación de Vectores de Refusal en LLMs', 
                 fontsize=14, fontweight='bold', y=0.98)
    plt.show()

visualizar_resumen_final()

## 6.3 Resumen de Fórmulas Clave

### Extracción de la dirección:

$$\mathbf{d}_{refusal} = \frac{1}{N}\sum_{i=1}^{N} \mathbf{h}^{harmful}_i - \frac{1}{M}\sum_{j=1}^{M} \mathbf{h}^{harmless}_j$$

### Ortogonalización de activaciones:

$$\mathbf{h}_{new} = \mathbf{h} - (\mathbf{h} \cdot \hat{\mathbf{d}}) \hat{\mathbf{d}}$$

### Ablación de matrices:

$$\mathbf{W}' = (\mathbf{I} - \hat{\mathbf{d}}\hat{\mathbf{d}}^T) \mathbf{W}$$

### Verificación:

$$\mathbf{h}_{new} \cdot \hat{\mathbf{d}} = 0 \quad \text{(sin componente de refusal)}$$

---

## 6.4 Código de Referencia

In [ ]:
# ============================================================
# CÓDIGO DE REFERENCIA COMPLETO
# ============================================================

class RefusalManipulator:
    """Clase para manipular vectores de refusal en LLMs"""
    
    def __init__(self):
        self.d_refusal = None
        self.d_refusal_per_layer = {}
    
    def extract_direction_mean_diff(self, harmful_activations, harmless_activations):
        """
        Extrae la dirección de refusal usando diferencia de medias.
        
        Args:
            harmful_activations: array (N, d) de activaciones de prompts harmful
            harmless_activations: array (M, d) de activaciones de prompts harmless
        
        Returns:
            d_refusal: vector normalizado (d,)
        """
        mean_harmful = np.mean(harmful_activations, axis=0)
        mean_harmless = np.mean(harmless_activations, axis=0)
        d = mean_harmful - mean_harmless
        self.d_refusal = d / np.linalg.norm(d)
        return self.d_refusal
    
    def extract_direction_pca(self, harmful_activations, harmless_activations, n_components=1):
        """
        Extrae direcciones de refusal usando PCA.
        
        Args:
            harmful_activations: array (N, d)
            harmless_activations: array (M, d)
            n_components: número de componentes principales a retornar
        
        Returns:
            directions: array (n_components, d)
        """
        all_data = np.vstack([harmful_activations, harmless_activations])
        centered = all_data - np.mean(all_data, axis=0)
        cov_matrix = np.cov(centered.T)
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # Ordenar por eigenvalue descendente
        idx = np.argsort(eigenvalues)[::-1]
        directions = eigenvectors[:, idx[:n_components]].T
        
        self.d_refusal = directions[0]  # Primera componente principal
        return directions
    
    def orthogonalize(self, h, d=None):
        """
        Ortogonaliza un vector h respecto a la dirección d.
        
        Args:
            h: vector de activación (d,) o batch (N, d)
            d: dirección de refusal (usa self.d_refusal si no se especifica)
        
        Returns:
            h_orth: vector ortogonalizado
        """
        if d is None:
            d = self.d_refusal
        d_norm = d / np.linalg.norm(d)
        
        if h.ndim == 1:
            return h - np.dot(h, d_norm) * d_norm
        else:
            # Batch
            projections = (h @ d_norm).reshape(-1, 1) * d_norm
            return h - projections
    
    def ablate_matrix(self, W, d=None):
        """
        Ablaciona una matriz W para que sus outputs no tengan componente en d.
        
        Args:
            W: matriz de pesos (out_dim, in_dim)
            d: dirección de refusal (usa self.d_refusal si no se especifica)
        
        Returns:
            W_ablated: matriz modificada
        """
        if d is None:
            d = self.d_refusal
        d_norm = d / np.linalg.norm(d)
        
        # Proyector ortogonal: I - d @ d.T
        P = np.eye(len(d_norm)) - np.outer(d_norm, d_norm)
        return P @ W
    
    def verify_orthogonality(self, h_orth, d=None, tolerance=1e-10):
        """
        Verifica que h_orth es ortogonal a d.
        
        Returns:
            bool: True si son ortogonales dentro de la tolerancia
        """
        if d is None:
            d = self.d_refusal
        d_norm = d / np.linalg.norm(d)
        dot_product = np.abs(np.dot(h_orth, d_norm))
        return dot_product < tolerance


# Demostración
print("🔧 Demostración del RefusalManipulator")
print("=" * 50)

# Crear datos de ejemplo
np.random.seed(42)
d_sim = 10  # Dimensión simulada

harmful = np.random.randn(50, d_sim) + np.array([1]*d_sim)  # Centrado en [1,1,...,1]
harmless = np.random.randn(50, d_sim) - np.array([1]*d_sim)  # Centrado en [-1,-1,...,-1]

# Usar el manipulador
manipulator = RefusalManipulator()

# 1. Extraer dirección
d = manipulator.extract_direction_mean_diff(harmful, harmless)
print(f"\\n1. Dirección de refusal extraída (primeras 5 dims): {d[:5]}")

# 2. Ortogonalizar un vector
h_test = np.random.randn(d_sim)
h_orth = manipulator.orthogonalize(h_test)
print(f"\\n2. Vector original · d: {np.dot(h_test, d):.4f}")
print(f"   Vector ortogonalizado · d: {np.dot(h_orth, d):.10f}")

# 3. Ablacionar una matriz
W_test = np.random.randn(d_sim, d_sim)
W_ablated = manipulator.ablate_matrix(W_test)

# Verificar que cualquier output de W_ablated es ortogonal a d
test_inputs = np.random.randn(20, d_sim)
outputs_original = test_inputs @ W_test.T
outputs_ablated = test_inputs @ W_ablated.T

print(f"\\n3. Componente medio en d de outputs de W original: {np.mean(np.abs(outputs_original @ d)):.4f}")
print(f"   Componente medio en d de outputs de W ablacionada: {np.mean(np.abs(outputs_ablated @ d)):.2e}")

# 4. Verificar ortogonalidad
is_orth = manipulator.verify_orthogonality(h_orth)
print(f"\\n4. ¿h_orth es ortogonal a d? {is_orth}")

print("\\n" + "=" * 50)
print("✅ Todas las operaciones funcionan correctamente!")

---

# 📚 Referencias y Recursos

## Papers relevantes:

1. **"Refusal in Language Models Is Mediated by a Single Direction"** (2024)
   - Demuestra que el refusal está codificado linealmente
   
2. **"Representation Engineering"** (Zou et al., 2023)
   - Técnicas para manipular representaciones internas

3. **"Linear Representations of Sentiment in Large Language Models"** (2023)
   - Muestra que conceptos como sentimiento son lineales

## Herramientas útiles:

- **TransformerLens**: Librería para interpretabilidad de transformers
- **nnsight**: Herramienta para intervenir en modelos
- **baukit**: Toolkit para análisis de neuronas

---

## Nota Ética

Este notebook es **educativo**. Entender cómo funcionan estas técnicas es importante para:

1. **Investigadores de seguridad**: Para desarrollar mejores defensas
2. **Equipos de alineación**: Para entender vulnerabilidades
3. **Educación**: Para comprender la interpretabilidad de LLMs

⚠️ **El uso irresponsable de estas técnicas puede tener consecuencias negativas.**

---

*Notebook creado para entender vectores de refusal en LLMs*